# Transfer Learning from ResNet50

In [2]:
import torch, torchvision
import numpy as np
from collect_data import collect_data, train_model

DATA_DIR = '../data/images'

# Load pretrained ResNet model
resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get training and validation data
train_loader, val_loader, test_loader = collect_data(DATA_DIR, preprocess)

1000 images processed
2000 images processed
3000 images processed
4000 images processed
5000 images processed
6000 images processed
7000 images processed
8000 images processed
9000 images processed
10000 images processed
11000 images processed
12000 images processed


In [2]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [3]:
class ResNetTransfer(torch.nn.Module):
    def __init__(self, resnetmodel):
        super(ResNetTransfer, self).__init__()
        # ResNet structure (without classifier)
        self.resnet = torch.nn.Sequential(resnetmodel.conv1,
                                            resnetmodel.bn1,
                                            resnetmodel.relu,
                                            resnetmodel.maxpool,
                                            resnetmodel.layer1,
                                            resnetmodel.layer2,
                                            resnetmodel.layer3,
                                            resnetmodel.layer4,
                                            resnetmodel.avgpool)
        # 2048 from the resnet convolution and 1 from the angles
        self.regressor = torch.nn.Sequential(torch.nn.Linear(2049, 1))
    
    def forward(self, X, angles):
        r = self.resnet(X)
        f = torch.flatten(r, 1)
        f = torch.cat([f, angles], dim=1)
        y = self.regressor(f)
        return y

In [4]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [6]:
import copy

model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.resnet.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 2050


In [5]:
# Hyperparameters
epsilon = 1E-5

# Use MSE loss
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device,
            num_epochs=1000, checkpoint_path='resnet_transfer_regressor_FULL.pth', graph_save_directory='./fully_trained_regressor')

Epoch [0/1000], Train Loss: 144.23561, Val Loss: 98.33664 *, Train Acc: 0.03595, Validation Acc: 0.01167
Epoch [1/1000], Train Loss: 122.03353, Val Loss: 79.19896 *, Train Acc: 0.04976, Validation Acc: 0.02611
Epoch [2/1000], Train Loss: 100.27945, Val Loss: 64.05436 *, Train Acc: 0.04714, Validation Acc: 0.03056
Epoch [3/1000], Train Loss: 86.67780, Val Loss: 51.57418 *, Train Acc: 0.04750, Validation Acc: 0.04000
Epoch [4/1000], Train Loss: 72.47726, Val Loss: 41.60955 *, Train Acc: 0.04810, Validation Acc: 0.05389
Epoch [5/1000], Train Loss: 65.04901, Val Loss: 34.04569 *, Train Acc: 0.04548, Validation Acc: 0.07444
Epoch [6/1000], Train Loss: 56.38763, Val Loss: 28.46438 *, Train Acc: 0.04512, Validation Acc: 0.08778
Epoch [7/1000], Train Loss: 49.23523, Val Loss: 24.46396 *, Train Acc: 0.05262, Validation Acc: 0.08833
Epoch [8/1000], Train Loss: 46.23259, Val Loss: 21.58307 *, Train Acc: 0.04940, Validation Acc: 0.07944
Epoch [9/1000], Train Loss: 42.35616, Val Loss: 19.80571 *, T

Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-2ce29f74-288d-42b0-8340-7eb3ac63ca4d.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-2ce29f74-288d-42b0-8340-7eb3ac63ca4d.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-2ce29f74-288d-42b0-8340-7eb3ac63ca4d.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-2ce29f74-288d-42b0-8340-7eb3ac63ca4d.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-2ce29f74-288d-42b0-8340-7eb3ac63ca4d.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connec

In [15]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_transfer_regressor_FULL.pth', criterion, test_loader, device)

Loss: 9.386364640129937
Accuracy: 0.1272222222222222
Off-by-One Accuracy: 0.37722222222222224


In [16]:
# Fine-tuning
finetune_model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)
finetune_model.load_state_dict(torch.load('resnet_transfer_regressor_FULL.pth', map_location=device))

for param in finetune_model.resnet.parameters():
    param.requires_grad = True

# Use different learning rate on different params to train resnet layers more slowly
finetune_optimizer = torch.optim.Adam([
    {'params': finetune_model.resnet.parameters(), 'lr': 1e-6},
    {'params': finetune_model.regressor.parameters(), 'lr': epsilon},
])

train_model(finetune_model, train_loader, val_loader, criterion, finetune_optimizer, device,
            num_epochs=100, checkpoint_path='resnet_finetuned_regressor_FULL.pth', graph_save_directory='./fully_trained_finetuned_regressor')

Epoch [0/100], Train Loss: 9.10195, Val Loss: 8.62025 *, Train Acc: 0.14238, Validation Acc: 0.12833
Epoch [1/100], Train Loss: 6.47432, Val Loss: 7.98484 *, Train Acc: 0.22560, Validation Acc: 0.12944
Epoch [2/100], Train Loss: 5.38160, Val Loss: 7.60002 *, Train Acc: 0.22619, Validation Acc: 0.13556
Epoch [3/100], Train Loss: 4.43727, Val Loss: 7.34669 *, Train Acc: 0.25476, Validation Acc: 0.14056
Epoch [4/100], Train Loss: 4.13326, Val Loss: 7.19984 *, Train Acc: 0.25536, Validation Acc: 0.13889
Epoch [5/100], Train Loss: 3.53381, Val Loss: 7.01220 *, Train Acc: 0.27893, Validation Acc: 0.13667
Epoch [6/100], Train Loss: 3.26293, Val Loss: 6.87985 *, Train Acc: 0.28155, Validation Acc: 0.14556
Epoch [7/100], Train Loss: 3.02367, Val Loss: 6.73987 *, Train Acc: 0.28929, Validation Acc: 0.14778
Epoch [8/100], Train Loss: 2.49157, Val Loss: 6.73911 *, Train Acc: 0.33714, Validation Acc: 0.14667
Epoch [9/100], Train Loss: 2.62966, Val Loss: 6.70514 *, Train Acc: 0.28512, Validation Acc

Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connec

In [18]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_finetuned_regressor_FULL.pth', criterion, test_loader, device)

Loss: 6.480497029622396
Accuracy: 0.14277777777777778
Off-by-One Accuracy: 0.45611111111111113


: 